# Scenario 2: Website Analytics Debug

### Prompt:

This Python function is supposed to calculate website engagement metrics, but it is giving impossible results like bounce rates over 100% and negative session times. Can you debug the calculations, add basic data validation, fix the time handling, and create a couple summary visualizations? Please explain what was wrong and what you changed.

In [ ]:
def analyze_user_engagement(logs_df):
    metrics = {
        'bounce_rate': logs_df.groupby('session_id')['page_views'].apply(
            lambda x: x == 1).mean(),
        'avg_session_time': logs_df.groupby('session_id')['duration'].sum(),
        'pages_per_session': logs_df.groupby('session_id')['page_views'].mean()
    }
    
    device_metrics = logs_df.groupby('device_type').agg({
        'session_id': 'count',
        'duration': 'mean',
        'page_views': 'sum'
    })
    
    return metrics, device_metrics

### Generated Code:

#### What was wrong

The function assumes one row per session, but a web log usually has **one row per page view or event**. Every metric was calculated at the wrong level.

| Metric | Problem | Effect |
|---|---|---|
| `bounce_rate` | `.apply(lambda x: x == 1)` gives one True/False **per log row**, not per session. `.mean()` then measures the share of *rows* with `page_views == 1`. | In a log with one row per page view, nearly every row has `page_views == 1`, so the bounce rate comes out near 100% whatever users actually do. Long sessions count many times. |
| `avg_session_time` | `.sum()` per session and never averaged. | Returns a list of session totals instead of one average. |
| `avg_session_time` | Sums the raw `duration` column without checking it. | Negative durations from upstream flow straight through. These usually come from calculating end − start when events are out of order, have swapped timestamps, or mix time zones. |
| `pages_per_session` | `.mean()` of page views within each session. | Averages rows (about 1 each) instead of adding up pages per session, and again returns a list, not one number. |
| `device_metrics` | `'session_id': 'count'` counts **rows**. `duration` is averaged per row. `page_views` is a grand total. | "Sessions" is really page-view count. Time is per page, not per session. Pages aren't per session. |
| Everything | No validation. | Duplicate hits double-count, blank session IDs merge into one fake session, and rows with a missing `device_type` silently disappear from the device table. |

**About "over 100%":** a mean of True/False values can't go above 1.0 on its own. Bounce rates over 100% appear when this row-based number is combined with session-based numbers, for example bounce *rows* divided by a *session* count, or when a value that is already a percentage is multiplied by 100 again. The rewrite computes bounce rate as bounced sessions ÷ sessions with at least one page view. By construction it always lands between 0% and 100%, and a check enforces that.

#### What changed

- **Session-level first:** the log is collapsed to one row per session: its device, total pages, and session time. Every metric, overall and by device, is then calculated from that table.
- **Validation:**
  - **Required columns:** all must be present, with a clear error naming any that are missing.
  - **Rows:** rows with no session ID are dropped. Exact duplicate events are dropped (only when timestamps exist, because without them two real page views can look identical).
  - **Device type:** normalized (`" Mobile"` → `mobile`), with missing values set to `unknown`.
  - **Page views:** missing, negative or fractional page views make that session's page count unknown instead of silently undercounting.
- **Time handling:**
  - **Timestamps are the preferred source.** They are parsed to UTC, which correctly handles mixed offsets such as `+02:00` alongside UTC times, and Unix epochs in seconds or milliseconds.
  - **Session time = last event − first event.** Using the latest minus the earliest time can't go negative, even when rows arrive out of order.
  - **`duration` as a fallback:** if there are no timestamps, the `duration` column (in seconds) is used, with negative values discarded.
  - **Runaway sessions:** sessions longer than `MAX_SESSION_MINUTES` (default 240) are treated as tracking errors, such as a tab left open or a bad clock.
- **Output:** one number per metric, plus the median session time, because a few long sessions distort the mean. The function returns the session table and a data-issues report as well.

**Known limit:** the first-to-last-event span doesn't include time spent on the final page, so single-page (bounced) sessions have 0 seconds. Every standard web-analytics tool has this limit unless the site sends a "page exit" event.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# --- Sample event log (one row per page view): replace with your own logs_df ---
rng = np.random.default_rng(7)
week_start = pd.Timestamp("2026-09-01")
rows = []
for i in range(400):
    device = rng.choice(["desktop", "mobile", "tablet"], p=[0.45, 0.45, 0.10])
    n_pages = 1 if rng.random() < 0.4 else int(rng.integers(2, 9))
    t = week_start + pd.Timedelta(minutes=float(rng.uniform(0, 60 * 24 * 7)))
    for _ in range(n_pages):
        dwell = float(rng.exponential(60))  # seconds on the page
        rows.append({"session_id": f"S{i:04d}", "device_type": device, "timestamp": t.isoformat(),
                     "page_views": 1, "duration": round(dwell, 1)})
        t += pd.Timedelta(seconds=dwell)
logs = pd.DataFrame(rows)

# Messy but realistic problems:
# 1. Some sessions logged by a server in UTC+2 (same instants, different offset)
berlin = logs["session_id"].isin([f"S{i:04d}" for i in range(10, 20)])
logs.loc[berlin, "timestamp"] = (pd.to_datetime(logs.loc[berlin, "timestamp"]) + pd.Timedelta(hours=2)
                                 ).dt.strftime("%Y-%m-%dT%H:%M:%S+02:00")
# 2. Inconsistent device labels
logs.loc[logs["session_id"].isin(["S0001", "S0002"]), "device_type"] = " Mobile"
# 3. Bad rows: missing session id, unparseable time, negative page views and duration, a runaway session
first = logs.groupby("session_id").first()  # device and start time of each generated session
bad = pd.DataFrame([
    {"session_id": None,    "device_type": "desktop",                    "timestamp": "2026-09-02T10:00:00",
     "page_views": 1,  "duration": 30},
    {"session_id": "S0003", "device_type": first.loc["S0003", "device_type"], "timestamp": "N/A",
     "page_views": 1,  "duration": 12},
    {"session_id": "S0004", "device_type": first.loc["S0004", "device_type"], "timestamp": first.loc["S0004", "timestamp"],
     "page_views": -1, "duration": 20},
    {"session_id": "S0005", "device_type": first.loc["S0005", "device_type"], "timestamp": first.loc["S0005", "timestamp"],
     "page_views": 1,  "duration": -45},
    {"session_id": "S9999", "device_type": None, "timestamp": "2026-09-04T08:00:00", "page_views": 1, "duration": 40},
    {"session_id": "S9999", "device_type": None, "timestamp": "2026-09-04T18:30:00", "page_views": 1, "duration": 40},
])
# 4. Double-logged hits, then shuffle so rows are out of time order
logs = pd.concat([logs, logs.sample(15, random_state=2), bad], ignore_index=True)
logs = logs.sample(frac=1, random_state=3).reset_index(drop=True)
logs.head()

In [ ]:
REQUIRED_COLUMNS = ["session_id", "device_type", "page_views"]
MAX_SESSION_MINUTES = 240  # longer sessions are treated as tracking errors (tab left open, bad clock)


def _to_float(s):
    num = pd.to_numeric(s, errors="coerce")
    return pd.Series(num.to_numpy(dtype="float64", na_value=np.nan), index=s.index).replace([np.inf, -np.inf], np.nan)


def _parse_timestamps(s):
    """Parse event times to naive UTC. Handles ISO strings with mixed offsets and Unix epochs (s or ms)."""
    if pd.api.types.is_numeric_dtype(s):
        unit = "ms" if s.abs().median() > 1e11 else "s"  # epoch ms values are ~1e12, seconds ~1e9
        ts = pd.to_datetime(s, unit=unit, errors="coerce", utc=True)
    else:
        try:
            ts = pd.to_datetime(s, errors="coerce", utc=True, format="mixed")
        except (TypeError, ValueError):  # pandas < 2.0 has no format="mixed"
            ts = pd.to_datetime(s, errors="coerce", utc=True)
    return ts.dt.tz_localize(None)


def validate_logs(logs_df):
    """Check columns and clean event-level values. Returns (clean_events, issues)."""
    if not isinstance(logs_df, pd.DataFrame):
        raise TypeError(f"Expected a pandas DataFrame, got {type(logs_df).__name__}")
    df = logs_df.rename(columns=lambda c: str(c).strip().lower())
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required column(s): {missing}. Found: {list(df.columns)}")
    if "timestamp" not in df.columns and "duration" not in df.columns:
        raise ValueError("Need a 'timestamp' column (preferred) or a 'duration' column in seconds to measure session time")
    if df.empty:
        raise ValueError("The log has no rows")

    df = df.copy()
    issues = {}

    df["session_id"] = df["session_id"].astype("string").str.strip().fillna("").astype(object)
    no_session = df["session_id"] == ""
    issues["rows with no session_id (dropped)"] = int(no_session.sum())
    df = df[~no_session]

    df["device_type"] = df["device_type"].astype("string").str.strip().str.lower().fillna("").astype(object)
    issues["rows with no device_type (set to 'unknown')"] = int((df["device_type"] == "").sum())
    df.loc[df["device_type"] == "", "device_type"] = "unknown"

    pv = _to_float(df["page_views"])
    bad_pv = pv.isna() | (pv < 0) | (pv % 1 != 0)
    issues["page_views missing, negative or fractional (session page count unknown)"] = int(bad_pv.sum())
    df["page_views"] = pv.mask(bad_pv)

    if "timestamp" in df.columns:
        df["timestamp"] = _parse_timestamps(df["timestamp"])
        issues["unparseable timestamps (row ignored for timing)"] = int(df["timestamp"].isna().sum())
        # Exact duplicates are double-logged hits. Only safe to drop when rows carry a timestamp;
        # without one, two genuine page views can look identical.
        dupes = df.duplicated()
        issues["duplicate events (dropped)"] = int(dupes.sum())
        df = df[~dupes]

    if "duration" in df.columns:
        dur = _to_float(df["duration"])
        issues["negative or non-numeric duration (set to NaN)"] = int((dur.isna() | (dur < 0)).sum())
        df["duration"] = dur.mask(dur < 0)

    if df.empty:
        raise ValueError("No usable rows left after validation")
    return df, issues


def build_sessions(events, max_session_minutes=MAX_SESSION_MINUTES):
    """Collapse the event log to one row per session."""
    g = events.groupby("session_id")
    sessions = pd.DataFrame({
        # a session is assigned its most common device; ties go to the alphabetically first
        "device_type": g["device_type"].agg(lambda s: s.mode().iat[0]),
        "events": g.size(),
        "pages": g["page_views"].sum(min_count=1),
    })
    # One bad page_views value makes the session's total unknown, rather than silently too low
    sessions.loc[events["page_views"].isna().groupby(events["session_id"]).any(), "pages"] = np.nan

    if "timestamp" in events.columns:
        # max - min is independent of row order, so it can never be negative
        sessions["start"] = g["timestamp"].min()
        sessions["session_seconds"] = (g["timestamp"].max() - sessions["start"]).dt.total_seconds()
    else:
        sessions["session_seconds"] = g["duration"].sum(min_count=1)
        sessions.loc[events["duration"].isna().groupby(events["session_id"]).any(), "session_seconds"] = np.nan

    too_long = sessions["session_seconds"] > max_session_minutes * 60
    sessions["session_seconds"] = sessions["session_seconds"].mask(too_long)
    sessions["session_minutes"] = sessions["session_seconds"] / 60

    # Bounce = exactly one page view. Sessions with 0 or unknown pages are excluded, not counted as non-bounces.
    has_pages = sessions["pages"] >= 1
    sessions["pages_valid"] = sessions["pages"].where(has_pages)
    sessions["bounced"] = (sessions["pages"] == 1).astype(float).where(has_pages)
    return sessions, int(too_long.sum())


def analyze_user_engagement(logs_df, max_session_minutes=MAX_SESSION_MINUTES):
    """Session-level engagement metrics.

    Returns (metrics, device_metrics, sessions, issues):
      metrics         dict of single numbers for the whole site
      device_metrics  one row per device type
      sessions        one row per session (for charts or drill-down)
      issues          counts of data problems found and how they were handled
    """
    events, issues = validate_logs(logs_df)
    sessions, n_too_long = build_sessions(events, max_session_minutes)
    issues[f"sessions over {max_session_minutes} min (time set to NaN)"] = n_too_long

    metrics = {
        "sessions": len(sessions),
        "bounce_rate": sessions["bounced"].mean(),
        "avg_session_minutes": sessions["session_minutes"].mean(),
        "median_session_minutes": sessions["session_minutes"].median(),
        "pages_per_session": sessions["pages_valid"].mean(),
        "time_source": "timestamps (first to last event)" if "timestamp" in events.columns else "duration column",
    }
    device_metrics = (sessions.groupby("device_type")
                      .agg(sessions=("events", "size"),
                           bounce_rate=("bounced", "mean"),
                           avg_session_minutes=("session_minutes", "mean"),
                           median_session_minutes=("session_minutes", "median"),
                           pages_per_session=("pages_valid", "mean"))
                      .sort_values("sessions", ascending=False))

    # Guard rails: these can't fail if the logic is right
    br = pd.concat([pd.Series([metrics["bounce_rate"]]), device_metrics["bounce_rate"]]).dropna()
    assert br.between(0, 1).all(), "bounce rate outside 0-100%"
    assert (sessions["session_seconds"].dropna() >= 0).all(), "negative session time"
    assert device_metrics["sessions"].sum() == len(sessions), "each session must belong to exactly one device"

    return metrics, device_metrics, sessions, issues

In [ ]:
metrics, device_metrics, sessions, issues = analyze_user_engagement(logs)

print("Data issues found:")
for issue, count in issues.items():
    if count:
        print(f"  {count:>4}  {issue}")

print(f"\nSessions:              {metrics['sessions']}")
print(f"Bounce rate:           {metrics['bounce_rate']:.1%}")
print(f"Avg session time:      {metrics['avg_session_minutes']:.1f} min (median {metrics['median_session_minutes']:.1f} min)")
print(f"Pages per session:     {metrics['pages_per_session']:.2f}")
print(f"Session time based on: {metrics['time_source']}")

display(device_metrics.style.format({
    "bounce_rate": "{:.1%}", "avg_session_minutes": "{:.1f}",
    "median_session_minutes": "{:.1f}", "pages_per_session": "{:.2f}",
}))

#### What the charts show

1. **Bounce rate by device.** This is the share of each device's sessions that viewed only one page. The dashed line is the site-wide rate. The session count next to each device name shows how much data sits behind each bar, so a small group like `unknown` isn't over-read. With the fix, every bar is between 0% and 100%.
2. **Session time by device.** This is the median minutes from first to last page view, for sessions with 2+ pages. Bounced sessions are left out because they always measure 0 minutes (there is no second event to time against). Including them would mostly just repeat chart 1. The median is used so a few very long sessions don't dominate.
3. **Pages per session.** This shows how many sessions viewed 1, 2, 3… pages. The dark bar at 1 page is the bounces, and the dashed line is the mean pages per session. A tall first bar with a long tail to the right is the normal shape for web traffic.

In [ ]:
DEVICE_COLORS = {"desktop": "#2a78d6", "mobile": "#eb6834", "tablet": "#1baf7a"}  # anything else is gray
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
STYLE = {
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.titlecolor": INK, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
}

dm = device_metrics.iloc[::-1]  # largest device group at the top
labels = [f"{d} (n={n})" for d, n in zip(dm.index, dm["sessions"])]
colors = [DEVICE_COLORS.get(d, "#898781") for d in dm.index]
# Bounced sessions have 0 min by definition (no second event), so time is shown for 2+ page sessions
engaged_time = (sessions[sessions["pages"] >= 2].groupby("device_type")["session_minutes"]
                .median().reindex(dm.index))

with plt.rc_context(STYLE):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17, 5))

    # (1) Bounce rate by device, with the site-wide rate for reference
    ax1.barh(labels, dm["bounce_rate"] * 100, color=colors, height=0.6)
    ax1.axvline(metrics["bounce_rate"] * 100, color=INK, lw=1, ls="--")
    ax1.text(metrics["bounce_rate"] * 100, len(dm) - 0.4, f" overall {metrics['bounce_rate']:.0%}",
             color=MUTED, fontsize=8, va="bottom")
    for y, v in enumerate(dm["bounce_rate"] * 100):
        if pd.notna(v):
            ax1.text(v, y, f" {v:.0f}%", va="center", color=INK)
    ax1.set(xlim=(0, 100), xlabel="Bounce rate (% of sessions with one page view)", title="Bounce rate by device")
    ax1.grid(axis="x", color=GRID, lw=0.6)

    # (2) Median session time by device, engaged (2+ page) sessions only
    ax2.barh(labels, engaged_time.fillna(0), color=colors, height=0.6)
    for y, v in enumerate(engaged_time):
        ax2.text(0 if pd.isna(v) else v, y, " n/a" if pd.isna(v) else f" {v:.1f} min", va="center", color=INK)
    ax2.set(xlabel="Median session time (minutes)", title="Session time by device (2+ page sessions)")
    ax2.grid(axis="x", color=GRID, lw=0.6)

    # (3) Distribution of pages per session; the 1-page bar is the bounces
    counts = sessions["pages_valid"].dropna().astype(int).value_counts().sort_index()
    ax3.bar(counts.index, counts.values, color="#86b6ef", width=0.8)
    ax3.bar(1, counts.get(1, 0), color="#184f95", width=0.8)
    ax3.text(1, counts.get(1, 0), "bounces", ha="center", va="bottom", color=MUTED, fontsize=8)
    ax3.axvline(metrics["pages_per_session"], color=INK, lw=1, ls="--")
    ax3.text(metrics["pages_per_session"], counts.max() * 0.95, f" mean {metrics['pages_per_session']:.1f}",
             color=MUTED, fontsize=8)
    ax3.set(xlabel="Pages viewed in the session", ylabel="Sessions", title="Pages per session",
            xticks=counts.index)
    ax3.grid(axis="y", color=GRID, lw=0.6)

    fig.tight_layout()
    plt.show()

### Follow-up Prompts:

This is a little more complicated than I need. Can you simplify it while still fixing the bounce rate, average session time, and pages per session calculations? Keep basic data validation, handle negative or invalid session times correctly, and include a few simple summary visualizations. I want the final code to be easy to read and focused on the main requirements.

### Final Solution:

**The fix in brief:** the log has one row per page view, but the original code calculated every metric per row instead of per session. This version first combines the rows into **one row per session**, then calculates:

- **Bounce rate** = sessions with exactly 1 page view ÷ all sessions. It can only be between 0% and 100%.
- **Average session time** = the average, across sessions, of each session's total `duration`. Negative or non-numeric durations are treated as invalid. A session with any invalid duration is left out of the time average instead of showing a time that's too short or negative.
- **Pages per session** = the average, across sessions, of each session's total page views.

**Validation:** the code checks that the required columns exist, drops rows with no `session_id`, cleans up device labels, and treats negative page views as missing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Sample data: one row per page view, duration = seconds spent on that page
rng = np.random.default_rng(7)
rows = []
for i in range(300):
    device = rng.choice(["desktop", "mobile", "tablet"], p=[0.45, 0.45, 0.10])
    n_pages = 1 if rng.random() < 0.4 else int(rng.integers(2, 8))
    for _ in range(n_pages):
        rows.append({"session_id": f"S{i:03d}", "device_type": device,
                     "page_views": 1, "duration": round(float(rng.exponential(60)), 1)})
logs = pd.DataFrame(rows)

# A few bad rows to show the validation working
bad_rows = pd.DataFrame({
    "session_id":  [None,      "S900",   "S901",    "S902"],
    "device_type": ["desktop", "mobile", " Mobile", None],
    "page_views":  [1,         1,        1,         1],
    "duration":    [30,        -45,      "n/a",     20],
})
logs = pd.concat([logs, bad_rows], ignore_index=True)

In [ ]:
REQUIRED_COLUMNS = ["session_id", "device_type", "page_views", "duration"]


def clean_logs(logs_df):
    """Basic validation: required columns, missing IDs, device labels, and invalid numbers."""
    missing = [c for c in REQUIRED_COLUMNS if c not in logs_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    has_id = logs_df["session_id"].notna() & (logs_df["session_id"].astype(str).str.strip() != "")
    df = logs_df[has_id].copy()

    df["device_type"] = df["device_type"].fillna("unknown").astype(str).str.strip().str.lower()
    df["page_views"] = pd.to_numeric(df["page_views"], errors="coerce")  # text -> NaN
    df["duration"] = pd.to_numeric(df["duration"], errors="coerce")

    # Negative page views or time are impossible, so treat them as missing
    df.loc[df["page_views"] < 0, "page_views"] = np.nan
    df.loc[df["duration"] < 0, "duration"] = np.nan
    return df


def analyze_user_engagement(logs_df):
    df = clean_logs(logs_df)

    # Step 1: one row per session
    sessions = df.groupby("session_id").agg(
        device_type=("device_type", "first"),
        pages=("page_views", "sum"),
        session_seconds=("duration", "sum"),
        has_invalid_time=("duration", lambda d: d.isna().any()),
    )
    # A session with any invalid duration gets no time, rather than a time that's too short
    sessions.loc[sessions["has_invalid_time"], "session_seconds"] = np.nan
    sessions = sessions[sessions["pages"] > 0].copy()  # a session needs at least one page view
    sessions["bounced"] = sessions["pages"] == 1

    # Step 2: metrics are averages across sessions, not across rows
    metrics = {
        "sessions": len(sessions),
        "bounce_rate": sessions["bounced"].mean(),
        "avg_session_time_sec": sessions["session_seconds"].mean(),  # NaN sessions are skipped
        "pages_per_session": sessions["pages"].mean(),
    }

    device_metrics = sessions.groupby("device_type").agg(
        sessions=("pages", "size"),
        bounce_rate=("bounced", "mean"),
        avg_session_time_sec=("session_seconds", "mean"),
        pages_per_session=("pages", "mean"),
    )
    return metrics, device_metrics, sessions

In [ ]:
metrics, device_metrics, sessions = analyze_user_engagement(logs)

print(f"Sessions:          {metrics['sessions']}")
print(f"Bounce rate:       {metrics['bounce_rate']:.1%}")
print(f"Avg session time:  {metrics['avg_session_time_sec'] / 60:.1f} min")
print(f"Pages per session: {metrics['pages_per_session']:.2f}")
print(f"Sessions left out of the time average (invalid duration): {sessions['session_seconds'].isna().sum()}")

device_metrics.round(2)

**What the charts show:**
1. **Bounce rate by device:** the percentage of each device's sessions that viewed only one page. The scale runs from 0 to 100%.
2. **Average session time by device:** the average minutes per session. Sessions with an invalid duration are left out.
3. **Pages per session:** how many sessions viewed 1, 2, 3… pages. The bar at 1 page is the bounces.

In [ ]:
COLORS = {"desktop": "#2a78d6", "mobile": "#eb6834", "tablet": "#1baf7a"}  # other devices are gray
bar_colors = [COLORS.get(d, "#898781") for d in device_metrics.index]

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. Bounce rate by device
ax1.bar(device_metrics.index, device_metrics["bounce_rate"] * 100, color=bar_colors)
ax1.set(title="Bounce rate by device", ylabel="% of sessions with 1 page", ylim=(0, 100))

# 2. Average session time by device
ax2.bar(device_metrics.index, device_metrics["avg_session_time_sec"] / 60, color=bar_colors)
ax2.set(title="Average session time by device", ylabel="Minutes")

# 3. How many pages each session viewed
page_counts = sessions["pages"].astype(int).value_counts().sort_index()
ax3.bar(page_counts.index, page_counts.values, color="#2a78d6")
ax3.set(title="Pages per session", xlabel="Pages viewed", ylabel="Number of sessions", xticks=page_counts.index)

for ax in (ax1, ax2, ax3):
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()